**Streaming With Langchain**

Importing the libraries

In [11]:
import asyncio
import langchain_core
import langchain_community
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

**Streaming with astream**

We will start by creating a aysnc stream from our LLM. We do this within an async for loop, allowing us to iterate through the chunks of data and use them as soon as the async astream method returns the tokens to us. By adding a pipe character | we can see the individual tokens that are generated. We set flush equal to True as this forces immediate output to the console, resulting in smoother streaming.

In [17]:
async def stream_tokens(prompt: str):
    tokens = []
    try:
        async for token in llm.astream(prompt):
            tokens.append(token)
            print(token.content, end="|", flush=True)
    except asyncio.CancelledError:
        print("\n\nStreaming was cancelled before the connection closed cleanly.")
    return tokens

tokens = await stream_tokens("What is NLP?")

N|LP| stands| for| Natural| Language| Processing|,| which| is| a| sub|field| of| artificial| intelligence| (|AI|)| that| deals| with| the| interaction| between| computers| and| humans| in| natural| language|.| It|'s| a| multid|isc|iplinary| field| that| combines| computer| science|,| lingu|istics|,| and| cognitive| psychology| to| enable| computers| to| understand|,| interpret|,| and| generate| human| language|.

|The| main| goals| of| N|LP| are|:

|1|.| **|Language| Understanding|**:| To| enable| computers| to| comprehend| the| meaning| of| text| or| speech|,| including| syntax|,| semantics|,| and| prag|m|atics|.
|2|.| **|Text| Analysis|**:| To| extract| insights| from| un|structured| text| data|,| such| as| sentiment| analysis|,| entity| recognition|,| and| topic| modeling|.
|3|.| **|Language| Generation|**:| To| generate| human|-like| text| or| speech| that| is| coherent| and| context|ually| relevant|.

|N|LP| has| many| applications| in| various| industries|,| including|:

|1|.| **

Since we appended each token to the tokens list, we can also see what is inside each and every token.

In [ ]:
tokens[0]

AIMessageChunk(content='N', additional_kwargs={}, response_metadata={}, id='lc_run--019e4b58-d161-7093-b07d-4bc6a577b763', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

In [ ]:
tokens[2]

AIMessageChunk(content=' stands', additional_kwargs={}, response_metadata={}, id='lc_run--019e4b58-d161-7093-b07d-4bc6a577b763', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

We can also merge multiple AIMessageChunk objects together with the + operator, creating a larger set of tokens / chunk:

In [ ]:
tokens[0] + tokens[1] + tokens[2] + tokens[3] + tokens[4]

AIMessageChunk(content='NLP stands for Natural', additional_kwargs={}, response_metadata={}, id='lc_run--019e4b58-d161-7093-b07d-4bc6a577b763', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

A word of caution, there is nothing preventing you from merging tokens in the incorrect order, so be cautious to not output any token omelettes:

In [ ]:
tokens[4] + tokens[3] + tokens[2] + tokens[1] + tokens[0]

AIMessageChunk(content=' Natural for standsLPN', additional_kwargs={}, response_metadata={}, id='lc_run--019e4b58-d161-7093-b07d-4bc6a577b763', tool_calls=[], invalid_tool_calls=[], tool_call_chunks=[])

**Streaming with Agents**

Streaming with agents, particularly the custom agent executor, is a little more complex. Let's begin by constructor a simple agent executor matching what we built in the Agent Executor chapter.

To construct the agent executor we need:

+ Tools
+ ChatPromptTemplate
+ Our LLM (already defined with llm)
+ An agent
+ Finally, the agent executor
+ Let's start defining each.

**Tools**
Now we will define a few tools to be used by an async agent executor. Our goal for tool-use in regards to streaming are:

+ The tool-use steps will be streamed in one big chunk, ie we do not return the tool use information token-by-token but instead it streams message-by-message.

+ The final LLM output will be streamed token-by-token as we saw above.

For these we need to define a few math tools and our final answer tool.

In [ ]:
from langchain_core.tools import tool

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`. You MUST use this tool
    to conclude the interaction.
    """
    return {"answer": answer, "tools_used": tools_used}


We'll need all of our tools in a list when defining our agent and agent_executor.

In [ ]:
tools = [add, multiply, exponentiate, subtract, final_answer]

**ChatPromptTemplate**

We will create our ChatPromptTemplate, using a system message, chat history, user input, and a scratchpad for intermediate steps.

In [13]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a helpful assistant. When answering a user's question "
        "you should first use one of the tools provided. After using a "
        "tool the tool output will be provided back to you. You MUST "
        "then use the final_answer tool to provide a final answer to the user. "
        "DO NOT use the same tool more than once."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

**Agent**

In [14]:
from langchain_core.runnables.base import RunnableSerializable

tools = [add, subtract, multiply, exponentiate, final_answer]

# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

**Agent Executor**

In [18]:
import json
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage


# create tool name to function mapping
name2tool = {tool.name: tool.func for tool in tools}

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    def invoke(self, input: str) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            out = self.agent.invoke({
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            })
            # if the model responds directly, return that answer safely
            if not out.tool_calls:
                final_answer = {"answer": out.content}
                final_answer_str = json.dumps(final_answer)
                self.chat_history.append({"input": input, "output": final_answer_str})
                self.chat_history.extend([
                    HumanMessage(content=input),
                    AIMessage(content=final_answer_str)
                ])
                return final_answer
            first_tool_call = out.tool_calls[0]
            # if the tool call is the final answer tool, we stop
            if first_tool_call["name"] == "final_answer":
                break
            agent_scratchpad.append(out)  # add tool call to scratchpad
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_out = name2tool[first_tool_call["name"]](**first_tool_call["args"])
            # add the tool output to the agent scratchpad
            action_str = f"The {first_tool_call['name']} tool returned {tool_out}"
            agent_scratchpad.append({
                "role": "tool",
                "content": action_str,
                "tool_call_id": first_tool_call["id"]
            })
            # add a print so we can see intermediate steps
            print(f"{count}: {action_str}")
            count += 1
        # add the final output to the chat history
        final_answer = first_tool_call["args"]
        # this is a dictionary, so we convert it to a string for compatibility with
        # the chat history
        final_answer_str = json.dumps(final_answer)
        self.chat_history.append({"input": input, "output": final_answer_str})
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer_str)
        ])
        # return the final answer in dict form
        return final_answer

agent_executor = CustomAgentExecutor()

Our agent_executor is now ready to use, let's quickly test it before adding streaming.

In [19]:
agent_executor.invoke(input="What is 10 + 10")

0: The add tool returned 20


{'answer': 'Final answer: The result of 10 + 10 is 20.'}

Let's modify our agent_executor to use streaming and parse the streamed output into a format that we can more easily work with.

First, when streaming with our custom agent executor we will need to pass our callback handler to the agent on every new invocation. To make this simpler we can make the callbacks field a configurable field and this will allow us to initialize the agent using the with_config method, allowing us to pass the callback handler to the agent with every invocation.

In [38]:
from langchain_core.runnables import ConfigurableField

llm = ChatOllama(
    model="llama3.1",
    temperature=0.0,
    streaming=True
).configurable_fields(
    callbacks=ConfigurableField(
        id="callbacks",
        name="callbacks",
        description="A list of callbacks to use for streaming",
    )
)

We reinitialize our agent, nothing changes here:

In [39]:
# define the agent runnable
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

Now, we will define our custom callback handler. This will be a queue callback handler that will allow us to stream the output of the agent through an asyncio.Queue object and yield the tokens as they are generated elsewhere.

In [40]:
import asyncio
from langchain_core.callbacks.base import AsyncCallbackHandler


class QueueCallbackHandler(AsyncCallbackHandler):
    """Callback handler that puts tokens into a queue."""

    def __init__(self, queue: asyncio.Queue):
        self.queue = queue
        self.final_answer_seen = False

    async def __aiter__(self):
        while True:
            if self.queue.empty():
                await asyncio.sleep(0.1)
                continue
            token_or_done = await self.queue.get()

            if token_or_done == "<<DONE>>":
                # this means we're done
                return
            if token_or_done:
                yield token_or_done

    async def on_llm_new_token(self, *args, **kwargs) -> None:
        """Put new token in the queue."""
        #print(f"on_llm_new_token: {args}, {kwargs}")
        chunk = kwargs.get("chunk")
        if chunk:
            # check for final_answer tool call
            if tool_calls := chunk.message.additional_kwargs.get("tool_calls"):
                if tool_calls[0]["function"]["name"] == "final_answer":
                    # this will allow the stream to end on the next `on_llm_end` call
                    self.final_answer_seen = True
        await self.queue.put(chunk)
        return

    async def on_llm_end(self, *args, **kwargs) -> None:
        """Put None in the queue to signal completion."""
        #print(f"on_llm_end: {args}, {kwargs}")
        # this should only be used at the end of our agent execution, however LangChain
        # will call this at the end of every tool call, not just the final tool call
        # so we must only send the "done" signal if we have already seen the final_answer
        # tool call
        if self.final_answer_seen:
            await self.queue.put("<<DONE>>")
        else:
            await self.queue.put("<<STEP_END>>")
        return

We can see how this works together in our agent invocation:

In [41]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

tokens = []

async def stream(query: str):
    response = agent.with_config(
        callbacks=[streamer]
    )
    async for token in response.astream({
        "input": query,
        "chat_history": [],
        "agent_scratchpad": []
    }):
        tokens.append(token)
        print(token, flush=True)

await stream("What is 10 + 10")

content='' additional_kwargs={} response_metadata={} id='lc_run--019e4bd4-be83-7c51-9a40-12d77b0929c8' tool_calls=[{'name': 'add', 'args': {'x': 10, 'y': 10}, 'id': 'e757ffc6-4a57-4849-bffe-7de2f8d05e56', 'type': 'tool_call'}] invalid_tool_calls=[] tool_call_chunks=[{'name': 'add', 'args': '{"x": 10, "y": 10}', 'id': 'e757ffc6-4a57-4849-bffe-7de2f8d05e56', 'index': None, 'type': 'tool_call_chunk'}]
content='' additional_kwargs={} response_metadata={'model': 'llama3.1', 'created_at': '2026-05-21T18:38:36.2757278Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22124646100, 'load_duration': 313885300, 'prompt_eval_count': 461, 'prompt_eval_duration': 18416965500, 'eval_count': 22, 'eval_duration': 3246741000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'} id='lc_run--019e4bd4-be83-7c51-9a40-12d77b0929c8' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 461, 'output_tokens': 22, 'total_tokens': 483} tool_call_chunks=[]
content='' additi

In [42]:
tk = tokens[0]

for token in tokens[1:]:
    tk += token

tk

AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-21T18:38:36.2757278Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22124646100, 'load_duration': 313885300, 'prompt_eval_count': 461, 'prompt_eval_duration': 18416965500, 'eval_count': 22, 'eval_duration': 3246741000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--019e4bd4-be83-7c51-9a40-12d77b0929c8', tool_calls=[{'name': 'add', 'args': {'x': 10, 'y': 10}, 'id': 'e757ffc6-4a57-4849-bffe-7de2f8d05e56', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 461, 'output_tokens': 22, 'total_tokens': 483}, tool_call_chunks=[{'name': 'add', 'args': '{"x": 10, "y": 10}', 'id': 'e757ffc6-4a57-4849-bffe-7de2f8d05e56', 'index': None, 'type': 'tool_call_chunk'}], chunk_position='last')

Now we're seeing that the output is being streamed token-by-token. Because we're being streamed a tool call the content field is empty. Instead, we can see the tokens being added inside the tool_calls fields, within id, function.name, and function.arguments.

In [47]:
from langchain_core.messages import ToolMessage

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    async def invoke(self, input: str, streamer: QueueCallbackHandler, verbose: bool = False) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            async def stream(query: str):
                response = self.agent.with_config(
                    callbacks=[streamer]
                )
                # we initialize the output dictionary that we will be populating with
                # our streamed output
                output = None
                first_tool_call = None
                # now we begin streaming
                async for token in response.astream({
                    "input": query,
                    "chat_history": self.chat_history,
                    "agent_scratchpad": agent_scratchpad
                }):
                    if output is None:
                        output = token
                    else:
                        # we can just add the tokens together as they are streamed and
                        # we'll have the full response object at the end
                        output += token
                    if token.content != "":
                        # we can capture various parts of the response object
                        if verbose: print(f"content: {token.content}", flush=True)
                    tool_calls = token.additional_kwargs.get("tool_calls")
                    if not tool_calls and token.tool_calls:
                        tool_calls = [
                            {
                                "function": {
                                    "name": token.tool_calls[0]["name"],
                                    "arguments": json.dumps(token.tool_calls[0]["args"])
                                },
                                "id": token.tool_calls[0]["id"]
                            }
                        ]
                    if tool_calls:
                        if first_tool_call is None and token.tool_calls:
                            first_tool_call = token.tool_calls[0]
                        if verbose: print(f"tool_calls: {tool_calls}", flush=True)
                        tool_name = tool_calls[0]["function"]["name"]
                        if tool_name:
                            if verbose: print(f"tool_name: {tool_name}", flush=True)
                        arg = tool_calls[0]["function"]["arguments"]
                        if arg != "":
                            if verbose: print(f"arg: {arg}", flush=True)
                if output is None:
                    return AIMessage(content="", tool_calls=[])
                if first_tool_call is not None:
                    return AIMessage(
                        content=output.content,
                        tool_calls=[first_tool_call],
                        tool_call_id=first_tool_call["id"]
                    )
                if output.tool_calls:
                    return AIMessage(
                        content=output.content,
                        tool_calls=output.tool_calls,
                        tool_call_id=output.tool_calls[0]["id"]
                    )
                return AIMessage(content=output.content, tool_calls=[])

            tool_call = await stream(query=input)
            if not tool_call.tool_calls:
                final_answer = tool_call.content
                self.chat_history.extend([
                    HumanMessage(content=input),
                    AIMessage(content=final_answer)
                ])
                await streamer.queue.put("<<DONE>>")
                return {"answer": final_answer}
            # add initial tool call to scratchpad
            agent_scratchpad.append(tool_call)
            # otherwise we execute the tool and add it's output to the agent scratchpad
            first_tool_call = tool_call.tool_calls[0]
            tool_name = first_tool_call["name"]
            tool_args = first_tool_call["args"]
            tool_call_id = tool_call.tool_call_id
            tool_out = name2tool[tool_name](**tool_args)
            # add the tool output to the agent scratchpad
            tool_exec = ToolMessage(
                content=f"{tool_out}",
                tool_call_id=tool_call_id
            )
            agent_scratchpad.append(tool_exec)
            count += 1
            # if the tool call is the final answer tool, we stop
            if tool_name == "final_answer":
                break
        # add the final output to the chat history, we only add the "answer" field
        final_answer = tool_out["answer"]
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])
        # return the final answer in dict form
        await streamer.queue.put("<<DONE>>")
        return tool_args

agent_executor = CustomAgentExecutor()

We've added a few print statements to help us see what is being output, we activate those by setting verbose=True. Let's see what is returned:


In [48]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

out = await agent_executor.invoke("What is 10 + 10", streamer, verbose=True)

tool_calls: [{'function': {'name': 'add', 'arguments': '{"x": 10, "y": 10}'}, 'id': 'b520cecb-7001-4cac-b74d-22bb5a2af738'}]
tool_name: add
arg: {"x": 10, "y": 10}
content: Final
content:  answer
content: :
content:  The
content:  result
content:  of
content:  
content: 10
content:  +
content:  
content: 10
content:  is
content:  
content: 20
content: .


We can see what is being output through the verbose=True flag. However, if we do not print the output, we will see nothing:

In [49]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

out = await agent_executor.invoke("What is 10 + 10", streamer)

Although we see nothing, it does not mean that nothing is being returned to us - we're just not using our callback handler and asyncio.Queue. To use these we create an asyncio task, iterate over the __aiter__ method of our streamer object, and await the task, like so:

In [50]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

task = asyncio.create_task(agent_executor.invoke("What is 10 + 10", streamer))

async for token in streamer:
    print(token, flush=True)

await task

message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={}, id='lc_run--019e4bd8-e192-7b22-ab44-aa844006940a', tool_calls=[{'name': 'add', 'args': {'x': 10, 'y': 10}, 'id': '522133d5-c490-4384-93a8-41dad6bd23cb', 'type': 'tool_call'}], invalid_tool_calls=[], tool_call_chunks=[{'name': 'add', 'args': '{"x": 10, "y": 10}', 'id': '522133d5-c490-4384-93a8-41dad6bd23cb', 'index': None, 'type': 'tool_call_chunk'}])
generation_info={'model': 'llama3.1', 'created_at': '2026-05-21T18:43:08.1018724Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22833873000, 'load_duration': 308172500, 'prompt_eval_count': 525, 'prompt_eval_duration': 19188393100, 'eval_count': 22, 'eval_duration': 3222400900, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'} message=AIMessageChunk(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-05-21T18:43:08.1018724Z', 'done': True, 'done_reason': 'stop', 'total_duration': 22833873

{'answer': 'Final answer: The result of 10 + 10 is 20.'}

Although this seems like a lot of work, we're now streaming tokens in a way that allows us to pass these tokens on to other parts of our code - such as through a websocket, streamed API response, or some downstream processing.

Let's try this out, we'll put together some simple post-processing to allow us to more nicely format the streamed output from out agent.

In [56]:
queue = asyncio.Queue()
streamer = QueueCallbackHandler(queue)

task = asyncio.create_task(agent_executor.invoke("What is 10 + 10", streamer))

async for token in streamer:
    # 1. Safely extract the inner message if it's wrapped in a ChatGenerationChunk
    msg = getattr(token, "message", token)
    
    if token == "<<STEP_END>>":
        print()
    
    # 2. Safely look for tool calls on the inner message
    elif (hasattr(msg, "tool_calls") and msg.tool_calls) or (hasattr(msg, "additional_kwargs") and msg.additional_kwargs.get("tool_calls")):
        print(f"\n[Calling Tool...]", flush=True)
        
    # 3. Safely grab the text content to stream to the screen
    elif hasattr(msg, "content") and msg.content:
        print(msg.content, end="", flush=True)

_ = await task


[Calling Tool...]

final_answer
The final answer is $\boxed{20}$.
